Set up local environment and folder for inputs and outputs

In [1]:
import os
import pathlib

# Set up local directories (no Google Colab)
base_dir = pathlib.Path.cwd()
dir_code = base_dir / 'Code/'
dir_inputs = base_dir / 'Input/'
dir_outputs = base_dir / 'Output/'

# Create directories if they don't exist
for d in [dir_inputs / 'COSIF/individual', dir_inputs / 'COSIF/prudencial', dir_inputs / 'IFDATA', dir_outputs]:
    d.mkdir(parents=True, exist_ok=True)

print(f"Base directory: {base_dir}")
print("Directories created successfully")

Base directory: c:\Github_offline\Credit Risk Modelling\Python
Directories created successfully


Downloading necessary library

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (classification_report, accuracy_score, 
                             precision_score, recall_score, confusion_matrix, 
                             precision_recall_curve, f1_score, roc_auc_score, 
                             roc_curve, auc)
from sklearn.metrics import roc_curve, auc
from statsmodels.stats.outliers_influence import variance_inflation_factor
import statsmodels.formula.api as smf
import statsmodels.api as sm
import pickle
from datetime import datetime
from scipy import interpolate
import requests
import zipfile
from pylab import rcParams
from matplotlib.ticker import FuncFormatter
import warnings

warnings.filterwarnings('ignore')

np.random.seed(42)
print("All imports loaded successfully")

All imports loaded successfully


In [3]:
pd.set_option('display.max_rows', None)

# COSIF
Define date paramenter to be used in automatic download. Dec/2022 and Jan/2023 have problem with format of COSIF files, and thus need to be downloaded in csv format.

In [4]:
date_range_csv = pd.date_range('2022-12-31','2023-01-31',
              freq='ME').strftime("%Y%m").tolist()
date_range_csv

['202212', '202301']

In [5]:
date_range_zip_1 = pd.date_range('2017-01-31','2022-11-30',
              freq='ME').strftime("%Y%m").tolist()
date_range_zip_1

['201701',
 '201702',
 '201703',
 '201704',
 '201705',
 '201706',
 '201707',
 '201708',
 '201709',
 '201710',
 '201711',
 '201712',
 '201801',
 '201802',
 '201803',
 '201804',
 '201805',
 '201806',
 '201807',
 '201808',
 '201809',
 '201810',
 '201811',
 '201812',
 '201901',
 '201902',
 '201903',
 '201904',
 '201905',
 '201906',
 '201907',
 '201908',
 '201909',
 '201910',
 '201911',
 '201912',
 '202001',
 '202002',
 '202003',
 '202004',
 '202005',
 '202006',
 '202007',
 '202008',
 '202009',
 '202010',
 '202011',
 '202012',
 '202101',
 '202102',
 '202103',
 '202104',
 '202105',
 '202106',
 '202107',
 '202108',
 '202109',
 '202110',
 '202111',
 '202112',
 '202201',
 '202202',
 '202203',
 '202204',
 '202205',
 '202206',
 '202207',
 '202208',
 '202209',
 '202210',
 '202211']

In [6]:
date_range_zip_2 = pd.date_range('2023-02-28','2026-03-31',
              freq='ME').strftime("%Y%m").tolist()
date_range_zip_2

['202302',
 '202303',
 '202304',
 '202305',
 '202306',
 '202307',
 '202308',
 '202309',
 '202310',
 '202311',
 '202312',
 '202401',
 '202402',
 '202403',
 '202404',
 '202405',
 '202406',
 '202407',
 '202408',
 '202409',
 '202410',
 '202411',
 '202412',
 '202501',
 '202502',
 '202503',
 '202504',
 '202505',
 '202506',
 '202507',
 '202508',
 '202509',
 '202510',
 '202511',
 '202512',
 '202601',
 '202602',
 '202603']

# COSIF - Individual bank level
Download COSIF in csv and zip format

In [7]:
for date in date_range_csv:
    url = "https://www.bcb.gov.br/content/estabilidadefinanceira/cosif/Bancos/" + date + "BANCOS.csv"
    output = dir_inputs / 'COSIF/individual' / f'{date}BANCOS.csv'
    r = requests.get(url)
    with open(output, 'wb') as f:
        f.write(r.content)
    print(f"Downloaded {date} CSV")

for date in date_range_zip_1:
    url = "https://www.bcb.gov.br/content/estabilidadefinanceira/cosif/Bancos/" + date + "BANCOS.zip"
    output = dir_inputs / 'COSIF/individual' / f'{date}BANCOS.zip'
    r = requests.get(url)
    with open(output, 'wb') as f:
        f.write(r.content)
    print(f"Downloaded {date} ZIP")

for date in date_range_zip_2:
    url = "https://www.bcb.gov.br/content/estabilidadefinanceira/cosif/Bancos/" + date + "BANCOS.csv.zip"
    output = dir_inputs / 'COSIF/individual' / f'{date}BANCOS.zip'
    r = requests.get(url)
    with open(output, 'wb') as f:
        f.write(r.content)
    print(f"Downloaded {date} CSV.ZIP")

Downloaded 202212 CSV
Downloaded 202301 CSV
Downloaded 201701 ZIP
Downloaded 201702 ZIP
Downloaded 201703 ZIP
Downloaded 201704 ZIP
Downloaded 201705 ZIP
Downloaded 201706 ZIP
Downloaded 201707 ZIP
Downloaded 201708 ZIP
Downloaded 201709 ZIP
Downloaded 201710 ZIP
Downloaded 201711 ZIP
Downloaded 201712 ZIP
Downloaded 201801 ZIP
Downloaded 201802 ZIP
Downloaded 201803 ZIP
Downloaded 201804 ZIP
Downloaded 201805 ZIP
Downloaded 201806 ZIP
Downloaded 201807 ZIP
Downloaded 201808 ZIP
Downloaded 201809 ZIP
Downloaded 201810 ZIP
Downloaded 201811 ZIP
Downloaded 201812 ZIP
Downloaded 201901 ZIP
Downloaded 201902 ZIP
Downloaded 201903 ZIP
Downloaded 201904 ZIP
Downloaded 201905 ZIP
Downloaded 201906 ZIP
Downloaded 201907 ZIP
Downloaded 201908 ZIP
Downloaded 201909 ZIP
Downloaded 201910 ZIP
Downloaded 201911 ZIP
Downloaded 201912 ZIP
Downloaded 202001 ZIP
Downloaded 202002 ZIP
Downloaded 202003 ZIP
Downloaded 202004 ZIP
Downloaded 202005 ZIP
Downloaded 202006 ZIP
Downloaded 202007 ZIP
Downloaded

Unzip COSIF

In [8]:
for date in date_range_zip_1:
    zip_path = dir_inputs / 'COSIF/individual' / f'{date}BANCOS.zip'
    extract_path = dir_inputs / 'COSIF/individual' / date
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(extract_path)
    print(f"Extracted {date} ZIP")

for date in date_range_zip_2:
    zip_path = dir_inputs / 'COSIF/individual' / f'{date}BANCOS.zip'
    extract_path = dir_inputs / 'COSIF/individual' / date
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(extract_path)
    print(f"Extracted {date} ZIP")

Extracted 201701 ZIP
Extracted 201702 ZIP
Extracted 201703 ZIP
Extracted 201704 ZIP
Extracted 201705 ZIP
Extracted 201706 ZIP
Extracted 201707 ZIP
Extracted 201708 ZIP
Extracted 201709 ZIP
Extracted 201710 ZIP
Extracted 201711 ZIP
Extracted 201712 ZIP
Extracted 201801 ZIP
Extracted 201802 ZIP
Extracted 201803 ZIP
Extracted 201804 ZIP
Extracted 201805 ZIP
Extracted 201806 ZIP
Extracted 201807 ZIP
Extracted 201808 ZIP


Extracted 201809 ZIP
Extracted 201810 ZIP
Extracted 201811 ZIP
Extracted 201812 ZIP
Extracted 201901 ZIP
Extracted 201902 ZIP
Extracted 201903 ZIP
Extracted 201904 ZIP


Extracted 201905 ZIP
Extracted 201906 ZIP


Extracted 201907 ZIP
Extracted 201908 ZIP
Extracted 201909 ZIP
Extracted 201910 ZIP
Extracted 201911 ZIP
Extracted 201912 ZIP
Extracted 202001 ZIP
Extracted 202002 ZIP
Extracted 202003 ZIP
Extracted 202004 ZIP
Extracted 202005 ZIP
Extracted 202006 ZIP
Extracted 202007 ZIP
Extracted 202008 ZIP
Extracted 202009 ZIP
Extracted 202010 ZIP
Extracted 202011 ZIP
Extracted 202012 ZIP
Extracted 202101 ZIP
Extracted 202102 ZIP


Extracted 202103 ZIP
Extracted 202104 ZIP
Extracted 202105 ZIP
Extracted 202106 ZIP
Extracted 202107 ZIP
Extracted 202108 ZIP


...

Extracted 202408 ZIP
Extracted 202409 ZIP
Extracted 202410 ZIP
Extracted 202411 ZIP
Extracted 202412 ZIP


Extracted 202501 ZIP
Extracted 202502 ZIP
Extracted 202503 ZIP
Extracted 202504 ZIP
Extracted 202505 ZIP
Extracted 202506 ZIP


Extracted 202507 ZIP
Extracted 202508 ZIP


Extracted 202509 ZIP
Extracted 202510 ZIP


Extracted 202511 ZIP
Extracted 202512 ZIP
Extracted 202601 ZIP
Extracted 202602 ZIP
Extracted 202603 ZIP


Merge all dates of COSIF invidual bank level

In [9]:
# Read each monthly file into a list and concatenate once.
# The original re-assigned `pd.concat([growing_frame, temp_df])` inside the loop, which copies the
# whole accumulated frame on every iteration - quadratic, ~37x slower here, and it leaves behind a
# duplicated garbage index (0..88649, repeated 111 times) instead of a clean RangeIndex.
#
# encoding: the COSIF bulk files are cp1252. 'unicode_escape' was used here before; it happens to
# agree for most values because it maps raw bytes to latin-1 code points, but it mangles byte 0x96 -
# the en dash in names like "Repasses do Pais - Instituicoes Oficiais", 3,508 occurrences across the
# panel - and would silently corrupt any value containing a backslash.
frames = []

# The .zip and .csv.zip eras differ only in the *download* extension; both extract to the same
# {YYYYMM}BANCOS.CSV member, so they load identically.
for date in date_range_zip_1 + date_range_zip_2:
    csv_path = dir_inputs / 'COSIF/individual' / date / f'{date}BANCOS.CSV'
    frames.append(pd.read_csv(csv_path, header=3, encoding='cp1252', on_bad_lines='skip',
                              sep=';', decimal=','))
    print(f"Loaded {date}")

# 202212 and 202301 are published unzipped.
for date in date_range_csv:
    csv_path = dir_inputs / 'COSIF/individual' / f'{date}BANCOS.csv'
    frames.append(pd.read_csv(csv_path, header=3, encoding='cp1252', on_bad_lines='skip',
                              sep=';', decimal=','))
    print(f"Loaded CSV {date}")

df_cosif_individual_full = pd.concat(frames, ignore_index=True)
print(f"Total COSIF individual records: {len(df_cosif_individual_full)}")

Loaded 201701

Loaded 201702
Loaded 201703
Loaded 201704


Loaded 201705


Loaded 201706
Loaded 201707
Loaded 201708


Loaded 201709


...

Loaded 202511


Loaded 202512
Loaded 202601


Loaded 202602
Loaded 202603
Loaded CSV 202212


Loaded CSV 202301


Total COSIF individual records: 2881939


Acquire COSIF account name

In [10]:
account_name = df_cosif_individual_full.sort_values('CONTA').drop_duplicates(subset=['CONTA'], keep='last')[['CONTA', 'NOME_CONTA']]

In [11]:
# BCB ships a stray control byte inside NOME_CONTA where a dash belongs: 0x1A in the individual
# files ("LIG, LCI e LCA EMITIDAS <0x1A> CONTROLE", account 3090400002) and 0x13 in the prudential
# ones ("Bens Arrendados <0x13> Arrendamento Financeiro"). This used to be patched with two
# hardcoded np.where() replacements keyed on the two account codes that happened to be affected in
# one vintage, so it silently stopped working as soon as BCB corrupted a different account.
# Strip any control character instead and collapse the whitespace it leaves behind - that
# reproduces the same two strings and keeps working for the rest.
CONTROL_CHARS = r'[\x00-\x1f\x7f-\x9f]'

dirty = account_name['NOME_CONTA'].str.contains(CONTROL_CHARS, regex=True, na=False)
print(f"Account names containing control characters: {int(dirty.sum())} "
      f"-> {account_name.loc[dirty, 'CONTA'].tolist()}")

account_name = account_name.assign(
    NOME_CONTA=account_name['NOME_CONTA']
        .str.replace(CONTROL_CHARS, ' ', regex=True)
        .str.replace(r'\s+', ' ', regex=True)
        .str.strip()
)

print(account_name.loc[dirty, ['CONTA', 'NOME_CONTA']].to_string(index=False))

Account names containing control characters: 2 -> [3090400002, 9097700008]
     CONTA                                  NOME_CONTA
3090400002            LIG, LCI e LCA EMITIDAS CONTROLE
9097700008 GARANTIAS EM ARRANJOS DE PAGAMENTO CONTROLE


In [12]:
account_name.to_excel(dir_outputs / 'cosif_account_name.xlsx')
print("Saved cosif_account_name.xlsx")

Saved cosif_account_name.xlsx


Reshape COSIF Individual bank level data from long format to wide formate

In [13]:
df_cosif_individual_long_format = df_cosif_individual_full.query('DOCUMENTO==4010')[['#DATA_BASE', 'CNPJ', 'NOME_INSTITUICAO', 'TAXONOMIA', 'CONTA', 'SALDO']]

In [14]:
df_cosif_individual_long_format['CONTA'] = df_cosif_individual_long_format['CONTA'].apply(str)

In [15]:
df_cosif_individual_long_format.rename(columns={'#DATA_BASE':'DATA'}, inplace=True)

In [16]:
df_cosif_individual_long_format.info()

<class 'pandas.core.frame.DataFrame'>
Index: 2376133 entries, 0 to 2881938
Data columns (total 6 columns):
 #   Column            Dtype  
---  ------            -----  
 0   DATA              int64  
 1   CNPJ              int64  
 2   NOME_INSTITUICAO  object 
 3   TAXONOMIA         object 
 4   CONTA             object 
 5   SALDO             float64
dtypes: float64(1), int64(2), object(3)
memory usage: 126.9+ MB


In [17]:
# One pivot over the whole panel instead of a per-period loop.
# DATA is part of the index, so the (index, CONTA) pairs are already unique across the full frame
# (verified: 0 duplicates) - the loop only existed because DataFrame.pivot needs a unique index, and
# it was accumulating with a quadratic pd.concat as well. Removing the loop also removes the
# hardcoded pd.date_range('2016-12-31', '2024-12-31') that used to bound it, which silently
# discarded every period from 202501 on: there is no date list left to go stale.
#
# Side effect: account columns now come out sorted by CONTA rather than in first-seen order. The
# fold step at the end of this notebook sorts them anyway, so df_final is unaffected.
df_cosif_individual = (df_cosif_individual_long_format
                       .pivot(index=['DATA', 'CNPJ', 'NOME_INSTITUICAO', 'TAXONOMIA'],
                              columns='CONTA', values='SALDO')
                       .rename_axis(None, axis=1)
                       .reset_index())

periods = sorted(df_cosif_individual['DATA'].unique())
print(f"Reshaped {len(periods)} periods ({periods[0]} to {periods[-1]}): "
      f"{df_cosif_individual.shape[0]} rows x {df_cosif_individual.shape[1]} columns")

Reshaped 111 periods (201701 to 202603): 19258 rows x 1265 columns


# COSIF Prudencial Conglomerate Data

In [18]:
for date in date_range_csv:
    url = "https://www.bcb.gov.br/content/estabilidadefinanceira/cosif/Conglomerados-prudenciais/" + date + "BLOPRUDENCIAL.csv"
    output = dir_inputs / 'COSIF/prudencial' / f'{date}PRUDENCIAL.csv'
    r = requests.get(url)
    with open(output, 'wb') as f:
        f.write(r.content)
    print(f"Downloaded prudential {date} CSV")

for date in date_range_zip_1:
    url = "https://www.bcb.gov.br/content/estabilidadefinanceira/cosif/Conglomerados-prudenciais/" + date + "BLOPRUDENCIAL.zip"
    output = dir_inputs / 'COSIF/prudencial' / f'{date}PRUDENCIAL.zip'
    r = requests.get(url)
    with open(output, 'wb') as f:
        f.write(r.content)
    print(f"Downloaded prudential {date} ZIP")

for date in date_range_zip_2:
    url = "https://www.bcb.gov.br/content/estabilidadefinanceira/cosif/Conglomerados-prudenciais/" + date + "BLOPRUDENCIAL.csv.zip"
    output = dir_inputs / 'COSIF/prudencial' / f'{date}PRUDENCIAL.zip'
    r = requests.get(url)
    with open(output, 'wb') as f:
        f.write(r.content)
    print(f"Downloaded prudential {date} CSV.ZIP")

Downloaded prudential 202212 CSV
Downloaded prudential 202301 CSV
Downloaded prudential 201701 ZIP
Downloaded prudential 201702 ZIP
Downloaded prudential 201703 ZIP
Downloaded prudential 201704 ZIP
Downloaded prudential 201705 ZIP
Downloaded prudential 201706 ZIP
Downloaded prudential 201707 ZIP
Downloaded prudential 201708 ZIP
Downloaded prudential 201709 ZIP
Downloaded prudential 201710 ZIP
Downloaded prudential 201711 ZIP
Downloaded prudential 201712 ZIP
Downloaded prudential 201801 ZIP
Downloaded prudential 201802 ZIP
Downloaded prudential 201803 ZIP
Downloaded prudential 201804 ZIP
Downloaded prudential 201805 ZIP
Downloaded prudential 201806 ZIP
Downloaded prudential 201807 ZIP
Downloaded prudential 201808 ZIP
Downloaded prudential 201809 ZIP
Downloaded prudential 201810 ZIP
Downloaded prudential 201811 ZIP
Downloaded prudential 201812 ZIP
Downloaded prudential 201901 ZIP
Downloaded prudential 201902 ZIP
Downloaded prudential 201903 ZIP
Downloaded prudential 201904 ZIP
Downloaded

Unzip COSIF Prudential Conglomerate Data

In [19]:
for date in date_range_zip_1:
    zip_path = dir_inputs / 'COSIF/prudencial' / f'{date}PRUDENCIAL.zip'
    extract_path = dir_inputs / 'COSIF/prudencial' / date
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(extract_path)
    print(f"Extracted prudential {date} ZIP")

for date in date_range_zip_2:
    zip_path = dir_inputs / 'COSIF/prudencial' / f'{date}PRUDENCIAL.zip'
    extract_path = dir_inputs / 'COSIF/prudencial' / date
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(extract_path)
    print(f"Extracted prudential {date} ZIP")

Extracted prudential 201701 ZIP
Extracted prudential 201702 ZIP
Extracted prudential 201703 ZIP
Extracted prudential 201704 ZIP
Extracted prudential 201705 ZIP
Extracted prudential 201706 ZIP
Extracted prudential 201707 ZIP
Extracted prudential 201708 ZIP
Extracted prudential 201709 ZIP
Extracted prudential 201710 ZIP
Extracted prudential 201711 ZIP
Extracted prudential 201712 ZIP
Extracted prudential 201801 ZIP
Extracted prudential 201802 ZIP
Extracted prudential 201803 ZIP
Extracted prudential 201804 ZIP
Extracted prudential 201805 ZIP
Extracted prudential 201806 ZIP
Extracted prudential 201807 ZIP
Extracted prudential 201808 ZIP
Extracted prudential 201809 ZIP
Extracted prudential 201810 ZIP
Extracted prudential 201811 ZIP
Extracted prudential 201812 ZIP
Extracted prudential 201901 ZIP
Extracted prudential 201902 ZIP
Extracted prudential 201903 ZIP
Extracted prudential 201904 ZIP
Extracted prudential 201905 ZIP
Extracted prudential 201906 ZIP
Extracted prudential 201907 ZIP
Extracte

Extracted prudential 202008 ZIP


Extracted prudential 202009 ZIP
Extracted prudential 202010 ZIP
Extracted prudential 202011 ZIP


Extracted prudential 202012 ZIP
Extracted prudential 202101 ZIP
Extracted prudential 202102 ZIP
Extracted prudential 202103 ZIP
Extracted prudential 202104 ZIP
Extracted prudential 202105 ZIP
Extracted prudential 202106 ZIP
Extracted prudential 202107 ZIP
Extracted prudential 202108 ZIP
Extracted prudential 202109 ZIP
Extracted prudential 202110 ZIP
Extracted prudential 202111 ZIP
Extracted prudential 202112 ZIP
Extracted prudential 202201 ZIP
Extracted prudential 202202 ZIP
Extracted prudential 202203 ZIP
Extracted prudential 202204 ZIP
Extracted prudential 202205 ZIP
Extracted prudential 202206 ZIP
Extracted prudential 202207 ZIP
Extracted prudential 202208 ZIP
Extracted prudential 202209 ZIP
Extracted prudential 202210 ZIP
Extracted prudential 202211 ZIP
Extracted prudential 202302 ZIP
Extracted prudential 202303 ZIP
Extracted prudential 202304 ZIP
Extracted prudential 202305 ZIP
Extracted prudential 202306 ZIP
Extracted prudential 202307 ZIP
Extracted prudential 202308 ZIP
Extracte

Extracted prudential 202410 ZIP
Extracted prudential 202411 ZIP
Extracted prudential 202412 ZIP


Extracted prudential 202501 ZIP
Extracted prudential 202502 ZIP
Extracted prudential 202503 ZIP
Extracted prudential 202504 ZIP
Extracted prudential 202505 ZIP
Extracted prudential 202506 ZIP
Extracted prudential 202507 ZIP
Extracted prudential 202508 ZIP
Extracted prudential 202509 ZIP


Extracted prudential 202510 ZIP


Extracted prudential 202511 ZIP
Extracted prudential 202512 ZIP
Extracted prudential 202601 ZIP
Extracted prudential 202602 ZIP
Extracted prudential 202603 ZIP


Merge all COSIF Prudential Conglomerate Data

In [20]:
# Same two fixes as the individual-level load: one concat instead of a quadratic accumulation,
# and cp1252 rather than 'unicode_escape'.
frames = []

for date in date_range_zip_1 + date_range_zip_2:
    csv_path = dir_inputs / 'COSIF/prudencial' / date / f'{date}BLOPRUDENCIAL.CSV'
    frames.append(pd.read_csv(csv_path, header=3, encoding='cp1252', on_bad_lines='skip',
                              sep=';', decimal=','))
    print(f"Loaded prudential {date}")

for date in date_range_csv:
    csv_path = dir_inputs / 'COSIF/prudencial' / f'{date}PRUDENCIAL.csv'
    frames.append(pd.read_csv(csv_path, header=3, encoding='cp1252', on_bad_lines='skip',
                              sep=';', decimal=','))
    print(f"Loaded prudential CSV {date}")

df_cosif_prudencial_full = pd.concat(frames, ignore_index=True)
print(f"Total COSIF prudential records: {len(df_cosif_prudencial_full)}")

Loaded prudential 201701


Loaded prudential 201702
Loaded prudential 201703
Loaded prudential 201704
Loaded prudential 201705


Loaded prudential 201706
Loaded prudential 201707
Loaded prudential 201708
Loaded prudential 201709
Loaded prudential 201710


Loaded prudential 201711
Loaded prudential 201712
Loaded prudential 201801
Loaded prudential 201802
Loaded prudential 201803


Loaded prudential 201804
Loaded prudential 201805
Loaded prudential 201806
Loaded prudential 201807
Loaded prudential 201808


...

Loaded prudential 202504
Loaded prudential 202505


Loaded prudential 202506
Loaded prudential 202507
Loaded prudential 202508


Loaded prudential 202509
Loaded prudential 202510
Loaded prudential 202511


Loaded prudential 202512
Loaded prudential 202601
Loaded prudential 202602


Loaded prudential 202603
Loaded prudential CSV 202212
Loaded prudential CSV 202301
Total COSIF prudential records: 2196580


Reshape COSIF Prudential Conglomerate data from long shape to wide shape

In [21]:
df_cosif_prudencial_long_format = df_cosif_prudencial_full.query('DOCUMENTO==4060')[['#DATA_BASE', 'CNPJ', 'NOME_INSTITUICAO',
                                                                                     'COD_CONGL', 'NOME_CONGL', 'TAXONOMIA', 'CONTA',
                                                                                     'NOME_CONTA', 'SALDO']]

In [22]:
df_cosif_prudencial_long_format['CONTA'] = df_cosif_prudencial_long_format['CONTA'].apply(str)

In [23]:
df_cosif_prudencial_long_format.rename(columns={'#DATA_BASE':'DATA'}, inplace=True)

In [24]:
# Same one-pass reshape as the individual level above (0 duplicate (index, CONTA) pairs here too).
df_cosif_prudencial = (df_cosif_prudencial_long_format
                       .pivot(index=['DATA', 'CNPJ', 'NOME_INSTITUICAO',
                                     'COD_CONGL', 'NOME_CONGL', 'TAXONOMIA'],
                              columns='CONTA', values='SALDO')
                       .rename_axis(None, axis=1)
                       .reset_index())

periods = sorted(df_cosif_prudencial['DATA'].unique())
print(f"Reshaped {len(periods)} periods ({periods[0]} to {periods[-1]}): "
      f"{df_cosif_prudencial.shape[0]} rows x {df_cosif_prudencial.shape[1]} columns")

Reshaped 111 periods (201701 to 202603): 14566 rows x 1336 columns


In [25]:
df_cosif_prudencial_important_var = df_cosif_prudencial[['DATA', 'CNPJ', 'COD_CONGL', 'NOME_CONGL']]

Merge COSIF Individual bank level data with important information (Prudential Conglomerate ID) of COSIF Prudencial Conglomerate Data

In [26]:
df_cosif = df_cosif_individual.merge(df_cosif_prudencial_important_var, on = ['DATA', 'CNPJ'], how='left')

Export COSIF complete data in parquet format

In [27]:
df_cosif.to_parquet(dir_outputs / 'df_cosif.parquet')
print("Saved df_cosif.parquet")

Saved df_cosif.parquet


### IF.DATA

Important account number:
conta = [79650, 79649, 79664, 79651, 79647, 79648, 79665, 79656]

Define date parameter to download IF.Data

In [28]:
date_range_quarter = pd.date_range('2017-01-31','2026-03-31',
              freq='QE').strftime("%Y%m").tolist()

In [29]:
for date in date_range_quarter:
    url = "https://olinda.bcb.gov.br/olinda/servico/IFDATA/versao/v1/odata/IfDataValores(AnoMes=@AnoMes,TipoInstituicao=@TipoInstituicao,Relatorio=@Relatorio)?@AnoMes="+date+"&@TipoInstituicao=1&@Relatorio='5'&$filter=Conta%20eq%20'79650'%20or%20Conta%20eq%20'79649'%20or%20Conta%20eq%20'79664'%20or%20Conta%20eq%20'79651'%20or%20Conta%20eq%20'79647'%20or%20Conta%20eq%20'79648'%20or%20Conta%20eq%20'79665'%20or%20Conta%20eq%20'79656'&$format=text/csv&$select=CodInst,AnoMes,Conta,NomeColuna,Saldo"
    output = dir_inputs / 'IFDATA' / f'{date}.csv'
    r = requests.get(url)
    with open(output, 'wb') as f:
        f.write(r.content)
    print(f"Downloaded IFDATA {date}")

Downloaded IFDATA 201703
Downloaded IFDATA 201706
Downloaded IFDATA 201709
Downloaded IFDATA 201712
Downloaded IFDATA 201803
Downloaded IFDATA 201806
Downloaded IFDATA 201809
Downloaded IFDATA 201812
Downloaded IFDATA 201903
Downloaded IFDATA 201906
Downloaded IFDATA 201909
Downloaded IFDATA 201912
Downloaded IFDATA 202003
Downloaded IFDATA 202006
Downloaded IFDATA 202009
Downloaded IFDATA 202012
Downloaded IFDATA 202103
Downloaded IFDATA 202106
Downloaded IFDATA 202109
Downloaded IFDATA 202112
Downloaded IFDATA 202203
Downloaded IFDATA 202206
Downloaded IFDATA 202209
Downloaded IFDATA 202212
Downloaded IFDATA 202303
Downloaded IFDATA 202306
Downloaded IFDATA 202309
Downloaded IFDATA 202312
Downloaded IFDATA 202403
Downloaded IFDATA 202406
Downloaded IFDATA 202409
Downloaded IFDATA 202412
Downloaded IFDATA 202503
Downloaded IFDATA 202506
Downloaded IFDATA 202509
Downloaded IFDATA 202512
Downloaded IFDATA 202603


Merge all months IF.Data

In [30]:
# The Olinda OData response is UTF-8. Reading it as 'unicode_escape' produced mojibake in
# NomeColuna ("Ãndice de Basileia" instead of "Índice de Basileia"). It went unnoticed because
# column_name() overwrites NomeColuna wholesale a few cells below - but nothing should depend on
# that, and the raw values are worth being able to read.
frames = []

for date in date_range_quarter:
    csv_path = dir_inputs / 'IFDATA' / f'{date}.csv'
    frames.append(pd.read_csv(csv_path, header=0, encoding='utf-8', on_bad_lines='skip',
                              sep=',', decimal=','))
    print(f"Loaded IFDATA {date}")

df_if_prudencial_full = pd.concat(frames, ignore_index=True)
print(f"Total IFDATA records: {len(df_if_prudencial_full)}")

Loaded IFDATA 201703
Loaded IFDATA 201706
Loaded IFDATA 201709
Loaded IFDATA 201712
Loaded IFDATA 201803
Loaded IFDATA 201806
Loaded IFDATA 201809
Loaded IFDATA 201812


Loaded IFDATA 201903


Loaded IFDATA 201906
Loaded IFDATA 201909
Loaded IFDATA 201912
Loaded IFDATA 202003
Loaded IFDATA 202006
Loaded IFDATA 202009


Loaded IFDATA 202012
Loaded IFDATA 202103
Loaded IFDATA 202106
Loaded IFDATA 202109
Loaded IFDATA 202112
Loaded IFDATA 202203
Loaded IFDATA 202206
Loaded IFDATA 202209
Loaded IFDATA 202212
Loaded IFDATA 202303


Loaded IFDATA 202306


Loaded IFDATA 202309
Loaded IFDATA 202312
Loaded IFDATA 202403
Loaded IFDATA 202406
Loaded IFDATA 202409
Loaded IFDATA 202412
Loaded IFDATA 202503
Loaded IFDATA 202506


Loaded IFDATA 202509
Loaded IFDATA 202512
Loaded IFDATA 202603
Total IFDATA records: 404202


Example of IF.Data of Banco do Brasil

In [31]:
df_if_prudencial_full.query('Conta == 79651 and AnoMes==201703 and CodInst == "C0080329" ')

,CodInst,AnoMes,Conta,NomeColuna,Saldo
8508,C0080329,201703,79651,RWA para Risco de Mercado \n(g) = (g1) + (g2) ...,9.722873e+09


Rename the name of important accounts containing information related to capital and risk-weighted asset

In [32]:
def column_name(row):
    if row['Conta'] == 79647:
        return "Tier_1"
    elif row['Conta'] == 79648:
        return "Tier_2"
    elif row['Conta'] == 79649:
        return "Capital"
    elif row['Conta'] == 79650:
        return "CRWA"
    elif row['Conta'] == 79651:
        return "MRWA"
    elif row['Conta'] == 79656:
        return "ORWA"
    elif row['Conta'] == 79664:
        return "BIS"
    else:
        return "RWA"

df_if_prudencial_full['NomeColuna'] = df_if_prudencial_full[['Conta']].apply(column_name, axis=1)

In [33]:
account_name_ifdata = df_if_prudencial_full.sort_values('Conta').drop_duplicates(subset=['Conta'], keep='last')[['Conta', 'NomeColuna']]
account_name_ifdata

,Conta,NomeColuna
404201,79647,Tier_1
245949,79648,Tier_2
404200,79649,Capital
1,79650,CRWA
404177,79651,MRWA
2,79656,ORWA
0,79664,BIS
404197,79665,RWA


In [34]:
account_name_ifdata.to_excel(dir_outputs / 'if_account_name.xlsx')
print("Saved if_account_name.xlsx")

Saved if_account_name.xlsx


In [35]:
df_if_prudencial_full = df_if_prudencial_full[['CodInst', 'AnoMes', 'Conta', 'NomeColuna', 'Saldo']]
df_if_prudencial_full['AnoMes'] = pd.to_numeric(df_if_prudencial_full['AnoMes'])
df_if_prudencial_full.rename(columns={'AnoMes': 'DATA', 'CodInst':'COD_CONGL'}, inplace=True)

Reshape IF.Data from long shape to wide shape

In [36]:
# A duplicate here means one institution reporting the *same account* twice in the same quarter,
# so the key has to include NomeColuna. This cell used to key on (DATA, COD_CONGL) alone, which
# flags 7 of every 8 rows purely because each institution legitimately reports 8 accounts per
# quarter - ~354k rows of noise that says nothing about data quality.
dup_key = ['DATA', 'COD_CONGL', 'NomeColuna']

# keep=False marks every member of a duplicated group, not just the repeats.
duplicate_mask = df_if_prudencial_full.duplicated(subset=dup_key, keep=False)
duplicate_rows = df_if_prudencial_full[duplicate_mask]

n_old_key = int(df_if_prudencial_full.duplicated(subset=['DATA', 'COD_CONGL']).sum())
print(f"Rows flagged by the old (DATA, COD_CONGL) key : {n_old_key:,} "
      f"(= 7/8 of {len(df_if_prudencial_full):,}, an artefact of 8 accounts per institution)")
print(f"Rows in a genuinely duplicated {tuple(dup_key)} group: {len(duplicate_rows):,}")

Rows flagged by the old (DATA, COD_CONGL) key : 354,199 (= 7/8 of 404,202, an artefact of 8 accounts per institution)
Rows in a genuinely duplicated ('DATA', 'COD_CONGL', 'NomeColuna') group: 28,875


In [37]:
df_if_prudencial_full.to_parquet(dir_outputs / 'df_if_prudencial_full.parquet')
print("Saved df_if_prudencial_full.parquet")

Saved df_if_prudencial_full.parquet


In [38]:
df_if_prudencial_full.sample(2)

,COD_CONGL,DATA,Conta,NomeColuna,Saldo
78057,C0084325,201809,79650,CRWA,981941.470000
52088,71154256,201803,79664,BIS,0.574451


In [39]:
# Where the duplication actually is, and whether dropping it can lose information.
if len(duplicate_rows):
    per_period = (duplicate_rows.groupby('DATA')
                  .agg(rows=('Saldo', 'size'), institutions=('COD_CONGL', 'nunique')))
    print("Duplicated rows by reporting period:")
    print(per_period.to_string())

    groups = duplicate_rows.groupby(dup_key)['Saldo']
    disagree = int((groups.nunique(dropna=False) > 1).sum())
    print(f"\nduplicate groups: {groups.ngroups:,}"
          f" | copies per group: {sorted(int(n) for n in groups.size().unique())}"
          f" | groups whose copies disagree on Saldo: {disagree}")
    print("-> every group is an exact copy, so keeping one member is lossless.")
else:
    print("No duplicated (DATA, COD_CONGL, NomeColuna) rows.")

Duplicated rows by reporting period:
         rows  institutions
DATA                       
202509  28875          1375

duplicate groups: 9,625 | copies per group: [3] | groups whose copies disagree on Saldo: 0
-> every group is an exact copy, so keeping one member is lossless.


In [40]:
# The 202509 Olinda response is triplicated upstream - every row appears three times,
# byte-identically (see the diagnostic above) - so the frame has to be deduplicated before pivoting
# or pandas raises on the non-unique index. DATA is part of the dedup key, so doing it once for the
# whole panel is exactly equivalent to the old per-quarter loop, and lossless.
df_if_prudencial = (df_if_prudencial_full
                    .drop_duplicates(subset=['DATA', 'COD_CONGL', 'NomeColuna'], keep='last')
                    .pivot(index=['DATA', 'COD_CONGL'], columns='NomeColuna', values='Saldo')
                    .rename_axis(None, axis=1)
                    .reset_index())

print(f"IF Prudencial data processed: {len(df_if_prudencial)} rows "
      f"x {df_if_prudencial.shape[1]} columns over "
      f"{df_if_prudencial['DATA'].nunique()} quarters")

IF Prudencial data processed: 50003 rows x 10 columns over 37 quarters


Export complete IF.Data in parquet format

---



In [41]:
df_if_prudencial.to_parquet(dir_outputs / 'df_if_prudencial.parquet')
print("Saved df_if_prudencial.parquet")

Saved df_if_prudencial.parquet


Merge COSIF and IF.Data

In [42]:
df_final = df_cosif.merge(df_if_prudencial, on=['DATA', 'COD_CONGL'], how='left')

# Additional Step: New accounting regulation in Brazil

### The "COSIF 1.5" update

CMN (National Monetary Council) **Resolution No. 4.966** took effect on **1 January 2025**. It
updates Brazil's standardised accounting framework for financial institutions (COSIF) to adopt
IFRS 9 principles, so historical losses give way to a forward-looking expected credit loss model.
Mechanically it also **renumbered the chart of accounts from 8-digit codes (COSIF 1.0) to 10-digit
codes (COSIF 1.5)**, so `10000007` became `1000000009`.

The reshape above pivots `CONTA` into columns, which means that renumbering leaves the account
columns **not comparable across the 202412 / 202501 break**: an 8-digit column is populated only up
to 202412, a 10-digit column only from 202501.

### Mapping

`map_cosif_final.xlsx` (sheet `final`) is the crosswalk between the two regimes. The fold below maps
COSIF 1.5 back onto COSIF 1.0 so that variable construction in the later notebooks is standardised
across both accounting regimes:

- `DATA <= 202412` -> keep the original COSIF 1.0 value
- `DATA >= 202501` -> replace it with the **sum** of all mapped COSIF 1.5 accounts

The mapping is many-to-one (several 1.5 accounts can roll up into one 1.0 account), which is why the
fold sums rather than renames. COSIF 1.0 accounts with no 1.5 counterpart pass through unchanged.

**Two things to know when reading the folded panel:**

1. The crosswalk was written against the full COSIF 1.5 chart of accounts, but the `BANCOS` bulk
   extract only publishes higher aggregation levels. Of the 225 distinct 1.5 codes in the map, only
   92 exist as columns here. So every mapping that resolves in this data is one-to-one, and the six
   many-to-one mappings (`16900008`, `17900007`, `18900006`, `31000000`, `31600008`, `31700001`) have
   **none** of their source accounts available and fall through to the pass-through branch.
2. Because of that, some accounts are simply unavailable from 202501 onward, including the asset
   quality (`A01`) and management (`M01`) CAMELS components. Check `.notna()` coverage per period
   before using an account in a model, rather than assuming the fold filled it.

In [43]:
import re


def fold_to_cosif10_sum_after_202501(
    df: pd.DataFrame,
    map_path,
    sheet_name: str = "final",
    data_col: str = "DATA",
    cutoff_last_old: int = 202412,   # <= this -> COSIF 1.0 value; > this -> sum of COSIF 1.5
    drop_cosif15: bool = True        # keep only COSIF 1.0 account columns
) -> pd.DataFrame:
    """
    Fold COSIF 1.5 (10-digit, CMN Res. 4.966, from 202501) account columns back onto their
    COSIF 1.0 (8-digit) equivalents, so one set of account columns spans both regimes.

    For each COSIF 1.0 account:
      - DATA <= cutoff_last_old : keep the original COSIF 1.0 value
      - DATA >  cutoff_last_old : replace with the sum of all mapped COSIF 1.5 columns
    COSIF 1.0 accounts with no mapping are kept and passed through unchanged.

    Note the mapping file stores codes with a literal 'X' prefix ('X10000007') while this
    notebook pivots CONTA into bare numeric strings ('10000007'). The prefix is stripped from
    the mapping rather than added to the dataframe, so df_final's column names stay exactly
    as the downstream notebooks (and the CAMELS account references) expect. Account columns
    are therefore identified by digit count, not by the 'X'.
    """
    COSIF10 = re.compile(r"^\d{8}$")    # COSIF 1.0 account code
    COSIF15 = re.compile(r"^\d{10}$")   # COSIF 1.5 account code

    # --- Load & sanitise the crosswalk ---
    m = pd.read_excel(map_path, sheet_name=sheet_name, dtype=str)
    m = m[[c for c in m.columns if c in ("Xcosif1_0", "Xcosif1_5")]].copy()
    m = m.map(lambda s: s.strip() if isinstance(s, str) else s)   # .applymap is removed in pandas 3
    for col in ("Xcosif1_0", "Xcosif1_5"):
        m[col] = m[col].str.removeprefix("X")                     # match the bare column names
    m = m.dropna(subset=["Xcosif1_0"])

    # COSIF 1.0 code -> list of every mapped COSIF 1.5 code
    m_group = (m.dropna(subset=["Xcosif1_5"])
                 .groupby("Xcosif1_0")["Xcosif1_5"]
                 .apply(list)
                 .to_dict())

    # --- Identify the COSIF 1.0 columns to keep ---
    x0_from_map = set(m["Xcosif1_0"])
    # 8-digit account columns in df that the crosswalk does not mention (pass-through)
    x0_no_map_in_df = {c for c in df.columns
                       if isinstance(c, str) and COSIF10.match(c) and c not in x0_from_map}
    x0_all = sorted(x0_from_map | x0_no_map_in_df)

    out = df.copy()

    # DATA must be numeric YYYYMM
    if not pd.api.types.is_integer_dtype(out[data_col]):
        out[data_col] = pd.to_numeric(out[data_col], errors="coerce").astype("Int64")
    is_old = (out[data_col] <= cutoff_last_old).to_numpy()

    folded, skipped = {}, []
    for x0 in x0_all:
        has_left = x0 in out.columns
        x5_present = [c for c in m_group.get(x0, []) if c in out.columns]
        if not has_left and not x5_present:
            # Neither the 1.0 column nor any mapped 1.5 column exists - creating this column
            # would just add an all-NaN placeholder, so skip it.
            skipped.append(x0)
            continue
        left_vals = out[x0] if has_left else pd.Series(np.nan, index=out.index)
        if x5_present:
            # min_count=1 keeps an all-NaN row as NaN instead of collapsing it to 0.0
            sum_15 = out[x5_present].sum(axis=1, skipna=True, min_count=1)
        else:
            sum_15 = left_vals   # no mapping -> pass through unchanged
        folded[x0] = np.where(is_old, left_vals, sum_15)

    # Assign in one pass (a per-column loop fragments the frame badly at this width)
    out = out.drop(columns=[c for c in folded if c in out.columns])
    out = pd.concat([out, pd.DataFrame(folded, index=out.index)], axis=1)

    # --- Keep non-account columns + COSIF 1.0 columns, drop the COSIF 1.5 columns ---
    x0_kept = [c for c in x0_all if c not in skipped]
    if drop_cosif15:
        non_account_cols = [c for c in out.columns
                            if not (isinstance(c, str)
                                    and (COSIF10.match(c) or COSIF15.match(c)))]
        out = out[non_account_cols + x0_kept]

    return out

In [44]:
# The crosswalk ships with the course material rather than being downloaded, so it lives in
# course_files/ (Input/ is gitignored and regenerable).
# The COSIF 1.0 -> 1.5 crosswalk ships with the course material.
def _find_ref(fname):
    """Course reference file: bundled in Input/reference/ for the standalone handover,
    else taken from the course_files/ checkout one level up."""
    for p in (base_dir / 'Input' / 'reference' / fname,
              base_dir.parent / 'course_files' / 'code' / fname):
        if p.exists():
            return p
    raise FileNotFoundError(fname)

map_path = _find_ref('map_cosif_final.xlsx')

df_merged = fold_to_cosif10_sum_after_202501(df_final, map_path=map_path, sheet_name="final")

print(f"Before fold: {df_final.shape[0]} rows x {df_final.shape[1]} columns")
print(f"After fold : {df_merged.shape[0]} rows x {df_merged.shape[1]} columns "
      f"(DATA {df_merged['DATA'].min()} to {df_merged['DATA'].max()})")

df_final = df_merged
df_final.info()

Before fold: 19258 rows x 1275 columns
After fold : 19258 rows x 262 columns (DATA 201701 to 202603)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 19258 entries, 0 to 19257
Columns: 262 entries, DATA to 99999995
dtypes: float64(256), int64(2), object(4)
memory usage: 38.5+ MB


In [45]:
df_final.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 19258 entries, 0 to 19257
Columns: 262 entries, DATA to 99999995
dtypes: float64(256), int64(2), object(4)
memory usage: 38.5+ MB


Export Final Database as df_final.parquet (main database for the whole course)

In [46]:
df_final.to_parquet(dir_outputs / 'df_final.parquet')
print("Saved df_final.parquet - Main database ready!")

Saved df_final.parquet - Main database ready!


# Additional Panel: Prudential Conglomerate Level

Everything above assembles the **individual bank** panel: COSIF `BANCOS` (`DOCUMENTO==4010`) pivoted
wide, with `COD_CONGL` borrowed from the prudential extract purely as a join key so IF.Data capital
and RWA can be attached.

That leaves a **consolidation-level mismatch** sitting at the heart of the dataset. The CAMELS
explanatory variables are individual-bank COSIF accounts, but `BIS`, `Capital`, `RWA` and `Tier_1/2`
come from IF.Data, which reports at the **prudential conglomerate** level. So every individual bank
inside a conglomerate inherits the *same* group-wide capital ratio, and since the target variable in
notebook 02 is a threshold on that ratio, it carries the *same* label as its siblings. Worse, an
institution belonging to no conglomerate gets no IF.Data at all and can't be labelled: only 53.9% of
the individual panel has an interpolated BIS ratio.

The cells below assemble the same data at the level the target is actually defined at.
`df_cosif_prudencial`, the full wide pivot of `DOCUMENTO==4060` built above and until now used only
for its `COD_CONGL`/`NOME_CONGL` columns, gets merged with IF.Data on `(DATA, COD_CONGL)` and run
through the same COSIF 1.5 fold.

No additional downloads: every file this needs is already in `Input/COSIF/prudencial/`.

See [`CHANGES_AND_WHY.md`](CHANGES_AND_WHY.md), change N1, for the full rationale, the measured
differences and what it costs.

In [47]:
# The prudential pivot is already in memory from the COSIF section above; it was only ever
# subset to ['DATA', 'CNPJ', 'COD_CONGL', 'NOME_CONGL'] for the individual-level merge.
print(f"df_cosif_prudencial: {df_cosif_prudencial.shape[0]:,} rows x "
      f"{df_cosif_prudencial.shape[1]} columns, "
      f"{df_cosif_prudencial['COD_CONGL'].nunique()} conglomerates")

# (DATA, COD_CONGL) is the natural key here - one row per conglomerate per month - and it has to be
# unique for the panel to make sense. Assert rather than assume.
n_dup = int(df_cosif_prudencial.duplicated(['DATA', 'COD_CONGL']).sum())
assert n_dup == 0, f"{n_dup} duplicate (DATA, COD_CONGL) rows"
print(f"duplicate (DATA, COD_CONGL) rows: {n_dup}")

df_congl = df_cosif_prudencial.merge(df_if_prudencial, on=['DATA', 'COD_CONGL'], how='left')
print(f"after IF.Data merge : {df_congl.shape[0]:,} rows x {df_congl.shape[1]} columns")
print(f"rows with a BIS value: {df_congl['BIS'].notna().sum():,} "
      f"({df_congl['BIS'].notna().mean():.1%}) - IF.Data is quarterly, so ~1 month in 3")
print(f"conglomerates matched: "
      f"{df_congl.loc[df_congl['BIS'].notna(), 'COD_CONGL'].nunique()} "
      f"of {df_congl['COD_CONGL'].nunique()}")

df_cosif_prudencial: 14,566 rows x 1336 columns, 230 conglomerates
duplicate (DATA, COD_CONGL) rows: 0
after IF.Data merge : 14,566 rows x 1344 columns
rows with a BIS value: 4,601 (31.6%) - IF.Data is quarterly, so ~1 month in 3
conglomerates matched: 221 of 230


In [48]:
# Same crosswalk, same function as the individual panel - the fold is level-independent.
df_final_congl = fold_to_cosif10_sum_after_202501(df_congl, map_path=map_path, sheet_name="final")

print(f"Before fold: {df_congl.shape[0]:,} rows x {df_congl.shape[1]} columns")
print(f"After fold : {df_final_congl.shape[0]:,} rows x {df_final_congl.shape[1]} columns "
      f"(DATA {df_final_congl['DATA'].min()} to {df_final_congl['DATA'].max()})")

Before fold: 14,566 rows x 1344 columns
After fold : 14,566 rows x 267 columns (DATA 201701 to 202603)


In [49]:
# What the level change buys, measured rather than asserted. Interpolation is done here only as a
# diagnostic - notebook 02 is where it actually happens.
def _bis_cover(df, key):
    filled = (df.sort_values([key, 'DATA'])
                .groupby(key)['Capital']
                .apply(lambda g: g.interpolate(method='linear', limit_area='inside'))
                .reset_index(level=0, drop=True))
    return df['BIS'].notna().mean(), filled.notna().mean()

ind_raw, ind_int = _bis_cover(df_final, 'CNPJ')
con_raw, con_int = _bis_cover(df_final_congl, 'COD_CONGL')

print(f"{'panel':<26}{'rows':>8}{'cols':>7}{'units':>8}{'BIS raw':>10}{'BIS interp':>12}")
print(f"{'individual (df_final)':<26}{len(df_final):>8,}{df_final.shape[1]:>7}"
      f"{df_final['CNPJ'].nunique():>8}{ind_raw:>10.1%}{ind_int:>12.1%}")
print(f"{'conglomerate':<26}{len(df_final_congl):>8,}{df_final_congl.shape[1]:>7}"
      f"{df_final_congl['COD_CONGL'].nunique():>8}{con_raw:>10.1%}{con_int:>12.1%}")

panel                         rows   cols   units   BIS raw  BIS interp
individual (df_final)       19,258    262     195     15.4%       46.1%
conglomerate                14,566    267     230     31.6%       92.6%


In [50]:
df_final_congl.to_parquet(dir_outputs / 'df_final_congl.parquet')
print(f"Saved df_final_congl.parquet - {df_final_congl.shape[0]:,} rows x "
      f"{df_final_congl.shape[1]} columns")
df_final_congl.info()

Saved df_final_congl.parquet - 14,566 rows x 267 columns
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 14566 entries, 0 to 14565
Columns: 267 entries, DATA to 99999995
dtypes: float64(261), int64(2), object(4)
memory usage: 29.7+ MB
